# Macro Higiênico

In [1]:
(define (square-f x)
  (* x x))

In [2]:
(square-f 10)

100

# `define-syntax`

## `define-rules`

In [3]:
(define-syntax square-m
  (syntax-rules ()
    [(square-m x)
     (* x x)]))

In [4]:
(square-m 10)

100

### Efeitos Indesejados

In [5]:
(begin (display "valor: ") 3)

valor: 3

In [6]:
(* (begin (display "valor: ") 3) 2)

valor: 6

## Tarefa

Escreva um exemplo que produza um efeito indesejado quando usado o square-m, mas não o produza quando usado o square-f.

## Por que é Higiênico

In [7]:
(define-macro (swap-n! a b)
  `(let ([temp ,a])
     (set! ,a ,b)
     (set! ,b temp)))

In [8]:
(let ([x 1]
      [y 2])
  (swap-n! x y)
  (list x y))

(2 1)

In [9]:
(let ([temp 1]
      [y 2])
  (swap-n! temp y)
  (list temp y))

(1 2)

In [10]:
(define-syntax swap-i!
  (syntax-rules ()
    [(swap-i! a b)
     (let ([temp a])
       (set! a b)
       (set! b temp))]))

In [11]:
(let ([x 1]
      [y 2])
  (swap-i! x y)
  (list x y))

(2 1)

In [12]:
(let ([temp 1]
      [y 2])
  (swap-i! temp y)
  (list temp y))

(2 1)

## `syntax-rules <literals>`

In [13]:
(define-syntax simple-path
  (syntax-rules (to)
    [(simple-path node1 to node2)
     '(node1 . node2)]))

In [14]:
(simple-path a to b)

(a . b)

## Explorando o potencial dos símbolos: `->`

In [15]:
(define-syntax simple-path
  (syntax-rules (->)
    [(simple-path node1 -> node2)
     '(node1 . node2)]))

In [16]:
(simple-path a -> b)

(a . b)

In [17]:
(define-syntax simple-path
  (syntax-rules (-> <-)
    [(simple-path node1 -> node2)
     '(node1 . node2)]
    [(simple-path node1 <- node2)
     '(node2 . node1)]))

In [18]:
(simple-path a -> b)

(a . b)

In [19]:
(simple-path a <- b)

(b . a)

In [20]:
(define-syntax simple-path
  (syntax-rules (-> <- <->)
    [(simple-path node1 -> node2)
     '(node1 . node2)]
    [(simple-path node1 <- node2)
     '(node2 . node1)]
    [(simple-path node1 <-> node2)
     '((node1 . node2) (node2 . node1))]))

In [21]:
(simple-path a <-> b)

((a . b) (b . a))

## `syntax-case`

In [22]:
(define-syntax simple-path
  (lambda (stx)
    (syntax-case stx (-> <- <->)

      [(simple-path node1 -> node2)
       #'(quote (node1 . node2))])))

In [23]:
(simple-path a -> b)

(a . b)

In [24]:
(define-syntax simple-path
  (lambda (stx)
    (syntax-case stx (-> <- <->)

      [(simple-path node1 -> node2)
       #'(quote (node1 . node2))]

      [(simple-path node1 <- node2)
       #'(quote (node2 . node1))]

      [(simple-path node1 <-> node2)
       #'(quote ((node1 . node2)
                 (node2 . node1)))])))

In [25]:
(simple-path a <-> b)

((a . b) (b . a))

## Além do pattern matching

In [26]:
(define-syntax simple-path
  (lambda (stx)
    (syntax-case stx (-> <- <->)

      [(simple-path node1 -> node2)
        (if (identifier? #'node2)
           #'(quote (node1 . node2))
           #'(quote ((node1 . $) ($ . node2))))])))

In [27]:
(simple-path a -> b)

(a . b)

In [28]:
(simple-path a -> 3)

((a . $) ($ . 3))

## `syntax->datum`

In [29]:
3

3

In [30]:
#'3

#<syntax 3

In [31]:
(syntax->datum #'3)

3

In [32]:
(define-syntax simple-path
  (lambda (stx)
    (syntax-case stx (-> <- <->)

      [(simple-path node1 -> node2)
        (if (number? (syntax->datum #'node2))
           #'(quote ((node1 . $) ($ . node2)))
           #'(quote (node1 . node2)))])))

In [33]:
(simple-path a -> 3)

((a . $) ($ . 3))